# Statistical Significance testing (for individual box and whisker plot comparsions)
This notebook will perform bootstrapping on the difference of means to determine if box-and-whisker plot pairs are statistically significantly different from one another.
This is done for means and extremes for one subregion at a time. Note some comparisons are meaningless (i.e. more than one difference in characteristics between pairs) and are excluded.

In [29]:
import numpy as np
import pandas as pd
from itertools import combinations

id = 1710 # Modify based on your desired subregion.
precip = pd.read_csv(f'intermediate_numpy_files/precip_for_sigtest_{id}.csv')
precip_extreme = precip[precip['precipitation'] >= precip.groupby(['era', 'ARDT', 'model'])['precipitation'].transform('quantile', 0.975)].dropna()

def bootstrap_diff_means(group_a, group_b, rng, n_boot=10000, ci=95, batch_size=2000):
    n_a, n_b = len(group_a), len(group_b)
    boot_diffs = np.empty(n_boot)
    
    for start in range(0, n_boot, batch_size):
        end = min(start + batch_size, n_boot)
        batch_n = end - start
        
        idx_a = rng.integers(0, n_a, size=(batch_n, n_a), dtype=np.int32)
        idx_b = rng.integers(0, n_b, size=(batch_n, n_b), dtype=np.int32)
        
        means_a = group_a[idx_a].mean(axis=1)
        means_b = group_b[idx_b].mean(axis=1)
        
        boot_diffs[start:end] = means_a - means_b
    
    observed_diff = group_a.mean() - group_b.mean()
    lower = np.percentile(boot_diffs, (100 - ci) / 2)
    upper = np.percentile(boot_diffs, 100 - (100 - ci) / 2)
    
    p_value = 2 * min(np.mean(boot_diffs <= 0), np.mean(boot_diffs >= 0))
    p_value = min(p_value, 1.0)
    
    return observed_diff, lower, upper, p_value

# Disclaimer: The function get_valid_pairs was AI-assisted. However, it has been verified by the author on functionality.
def get_valid_pairs(df, group_cols, excluded_value_pairs=None, bypass_rules=None):
    """
    excluded_value_pairs: dict col -> list of (val_a, val_b) pairs to exclude
        when that column is the sole difference.
    bypass_rules: dict col -> dict of (val_a, val_b) -> list of other columns
        allowed to also differ for that specific value pair.
        e.g. {"era": {("obs", "historical"): ["model"]}}
    
    Returns list of (combo1, combo2, diff_cols) tuples, where diff_cols
    is a list of the column names that actually differ between the pair.
    """
    excluded_value_pairs = excluded_value_pairs or {}
    bypass_rules = bypass_rules or {}
    combos = df[group_cols].drop_duplicates().apply(tuple, axis=1).tolist()
    
    valid_pairs = []
    for c1, c2 in combinations(combos, 2):
        diffs = [(col, a, b) for col, a, b in zip(group_cols, c1, c2) if a != b]
        
        if len(diffs) == 1:
            diff_col, val_a, val_b = diffs[0]
            forbidden = excluded_value_pairs.get(diff_col, [])
            if (val_a, val_b) in forbidden or (val_b, val_a) in forbidden:
                continue
            valid_pairs.append((c1, c2, [diff_col]))
        
        elif len(diffs) == 2:
            matched = False
            for diff_col, val_a, val_b in diffs:
                rules = bypass_rules.get(diff_col, {})
                allowed_cols = rules.get((val_a, val_b)) or rules.get((val_b, val_a))
                if allowed_cols:
                    other_cols = [d[0] for d in diffs if d[0] != diff_col]
                    if all(c in allowed_cols for c in other_cols):
                        valid_pairs.append((c1, c2, [d[0] for d in diffs]))
                        matched = True
                        break
            # if no bypass rule matched, this pair is skipped
        
        # 3+ diffs are always skipped
    
    return valid_pairs


def compare_specific_subgroups(df, value_col, group_cols, excluded_value_pairs=None,
                                 bypass_rules=None, n_boot=10000, ci=95, alpha=0.05,
                                 seed=42, batch_size=2000):
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    
    pairs = get_valid_pairs(df, group_cols,
                             excluded_value_pairs=excluded_value_pairs,
                             bypass_rules=bypass_rules)
    
    grouped = {
        name: sub[value_col].dropna().to_numpy()
        for name, sub in df.groupby(group_cols)
    }
    
    rng = np.random.default_rng(seed)
    n_comparisons = len(pairs)
    bonferroni_alpha = alpha / 24
    
    results = []
    for c1, c2, diff_cols in pairs:
        vals1, vals2 = grouped[c1], grouped[c2]
        diff, lower, upper, p_value = bootstrap_diff_means(
            vals1, vals2, rng, n_boot=n_boot, ci=ci, batch_size=batch_size
        )
        
        results.append({
            "group_1": c1,
            "group_2": c2,
            "differs_on": ", ".join(diff_cols),
            "n_1": len(vals1),
            "n_2": len(vals2),
            "observed_diff": diff,
            "ci_lower": lower,
            "ci_upper": upper,
            "p_value": p_value,
            "significant_raw": p_value < alpha,
            "significant_bonferroni": p_value < bonferroni_alpha,
        })
    
    results_df = pd.DataFrame(results)
    results_df.attrs["n_comparisons"] = n_comparisons
    results_df.attrs["bonferroni_alpha"] = bonferroni_alpha
    return results_df

# Perform the bootstrap on specific subgroup pairs only. Saves time and avoids meaningless comparisons.
excluded = {"era": [("obs", "near-term"), ("obs", "end-of-century")], "model": [('cmip6nonbc', 'cesm2le')]}
bypass_rules = {"era": {("obs", "historical"): ["model"]}}
results = compare_specific_subgroups(precip, value_col="precipitation", group_cols=["era", "model", "ARDT"], excluded_value_pairs=excluded, bypass_rules=bypass_rules)
results_extreme = compare_specific_subgroups(precip_extreme, value_col="precipitation", group_cols=["era", "model", "ARDT"], excluded_value_pairs=excluded, bypass_rules=bypass_rules)
print(f"Bonferroni-adjusted alpha: {results.attrs['bonferroni_alpha']:.5f}")

Bonferroni-adjusted alpha: 0.00208


In [24]:
results.to_csv(f'/glade/work/tcorrie/ARdata/ARwatershed/individual_statsigtesting_{id}.csv')
results_extreme.to_csv(f'/glade/work/tcorrie/ARdata/ARwatershed/individual_statsigtesting_extreme_{id}.csv')